# 2/2 · RadiologyBot AI : chatbot RAG (radiologie)

Ce notebook lit la base `chroma_db` créée par le notebook `creation_base` (via Google Drive) et répond aux questions avec Gemma.

**Prérequis :** avoir exécuté `creation_base` jusqu'au bout (le dossier `radiologie_rag/chroma_db` doit exister dans votre Drive).

## 1. Installer et démarrer Ollama

In [ ]:
# Installe zstd (nécessaire pour décompresser l'archive d'Ollama)
!sudo apt-get install -y zstd
# Télécharge et installe Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Lance le serveur Ollama en arrière-plan (systemd n'existe pas dans Colab)
!nohup ollama serve > ollama.log 2>&1 &

# Attend que le serveur soit prêt
import time
time.sleep(5)

In [ ]:
# Modèle d'embeddings (le même que pour la création de la base)
!ollama pull bge-m3
# Modèle de chat qui génère les réponses
!ollama pull gemma4:e4b
!ollama list

## 2. Installer les bibliothèques Python

In [ ]:
!pip install -q langchain-ollama langchain-chroma

## 3. Connecter Google Drive et récupérer la base

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

CHROMA_DRIVE = Path("/content/drive/MyDrive/radiologie_rag/chroma_db")
CHROMA_LOCAL = Path("/content/chroma_db")

if not CHROMA_DRIVE.exists():
    raise FileNotFoundError(
        "chroma_db introuvable dans Drive. "
        "Exécutez d'abord le notebook creation_base jusqu'au bout."
    )

# Copie la base sur le disque local de Colab (plus fiable que lire directement dans Drive)
shutil.rmtree(CHROMA_LOCAL, ignore_errors=True)
shutil.copytree(CHROMA_DRIVE, CHROMA_LOCAL)

print("Base récupérée :", [p.name for p in CHROMA_LOCAL.iterdir()])

## 4. Charger les modèles et la base

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma


# 1. Charger le modèle d'embeddings
embeddings = OllamaEmbeddings(
    model="bge-m3"
)


# 2. Charger la base de connaissances
vectorstore = Chroma(
    collection_name="radiologie",
    persist_directory="/content/chroma_db",
    embedding_function=embeddings
)


# 3. Créer le moteur de recherche
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)


# 4. Charger le modèle Gemma
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1
)

# Vérification : la base ne doit pas être vide
print("Passages dans la base :", vectorstore._collection.count())

## 5. Fonction principale du chatbot

In [ ]:
def ask_ai(question: str) -> str:

    # Rechercher les passages correspondant à la question
    documents = retriever.invoke(question)

    # Vérifier si des informations ont été trouvées
    if not documents:
        return (
            "Je ne trouve pas cette information dans "
            "les documents du service de radiologie. "
            "Veuillez contacter l'accueil de l'hôpital."
        )

    # Récupérer le contenu des passages
    contexte = "\n\n".join(
        document.page_content
        for document in documents
    )

    # Construire le prompt
    prompt = f"""
Tu es RadiologyBot AI, un assistant d'information
du service de radiologie de l'hôpital régional de Thiès.

Ta mission est de répondre aux questions des patients.

RÈGLES IMPORTANTES :
- Utilise uniquement les informations du contexte.
- N'invente jamais d'horaires, de tarifs ou de prestations.
- Si la réponse n'est pas dans le contexte, dis-le clairement.
- Ne donne pas de diagnostic médical.
- Oriente le patient vers le personnel hospitalier
  pour toute question médicale personnelle.
- Réponds en français simple et compréhensible.

CONTEXTE :
{contexte}

QUESTION DU PATIENT :
{question}

RÉPONSE :
"""

    # Envoyer le prompt à Gemma
    reponse = llm.invoke(prompt)

    return reponse.content

## 6. Tester le chatbot

In [ ]:
# Question dont la réponse est dans le document
print(ask_ai("Quels sont les horaires du service de radiologie ?"))

In [ ]:
# Question dont la réponse est dans le document
print(ask_ai("Quels documents dois-je apporter ?"))

In [ ]:
# Question dont la réponse N'EST PAS dans le document : le bot doit dire qu'il ne sait pas
print(ask_ai("Combien coûte une mammographie ?"))

## 7. Discuter en continu (optionnel)

Tapez `quit` pour arrêter la boucle.

In [ ]:
while True:
    question = input("Vous : ")
    if question.strip().lower() in ("quit", "exit", "q"):
        break
    print("RadiologyBot :", ask_ai(question), "\n")